# E12 — is the part effect real, and is it additive?

Two questions, in order, because the second is meaningless if the first fails.

1. **Does `u_p` replicate?** Fit the part-effect matrix on one population, fit it again on a
   second independent one, compare. If the two disagree, there is no part effect to decompose.
2. **Does it superpose?** Compare `u_p` measured alone (`n_active=1`) against `u_p` measured in
   company (`n_active=k`). That is the P2 §2 equation, tested directly.

**What we already know, and why the σ column is the whole story.** Expand the per-example score:

$$q_{ib} = q_b + \langle g_b, E_i\rangle + \tfrac12 E_i^\top H_b E_i + O(\sigma^3), \qquad E_i = \sigma\,(M_i \odot Z_i)$$

Condition on member $i$ having perturbed part $p$ and average over the direction $Z$. The
first-order term **vanishes** — $\mathbb{E}[\langle g_b, E_i\rangle \mid p] = 0$ because $Z$ is
zero-mean whatever the mask is. Part identity survives only in the curvature term:

$$\mathrm{SNR}(\hat u_p)\;\propto\;\sigma\sqrt{n}\;\frac{\mathrm{tr}(H_b|_p)}{\lVert g_b|_p\rVert}$$

$U$ is a **curvature object estimated against gradient noise**. Signal $O(\sigma^2)$, noise sd
$O(\sigma)$. At the training σ = 0.005 it is not estimable at any population size you can afford;
at σ ≈ 0.05–0.2 it is. That is what these cells measure, and it is why the design that follows
needs a separate probe phase at inflated σ.

**Runtime.** Steps 1–2 are the real content. On a Colab **T4** the defaults take ~10–20 min; on
**CPU** budget 45–90 min, or drop `N_POP` to `256` and `REPEATS` to `1`.

## Setup

In [ ]:
import torch, sys, platform
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} · python {platform.python_version()} · device {DEVICE}")
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))
else:
    print("No GPU. Runtime > Change runtime type > T4 GPU makes this ~4x faster.")
    print("CPU is the verified path; the GPU path is the same code with --device cuda.")

### Get the code

`TomOffermann/evolve` is private, so a plain `git clone` will 404. Use a fine-grained PAT with
**Contents: read** on that one repo. The token is read with `getpass` (not echoed) and stripped
from the git remote immediately after, so it does not persist in the notebook or in `.git/config`.

Alternatives if you would rather not paste a token: mount Drive and point `REPO` at a synced copy,
or zip the repo and use the Colab file uploader. Both are in the commented block.

In [ ]:
import os, subprocess
from pathlib import Path
from getpass import getpass

REPO = Path("/content/evolve")

if not REPO.exists():
    token = getpass("GitHub PAT (Contents:read on TomOffermann/evolve, input hidden): ").strip()
    url = f"https://{token}@github.com/TomOffermann/evolve.git"
    r = subprocess.run(["git", "clone", "--depth", "1", url, str(REPO)],
                       capture_output=True, text=True)
    print(r.stdout or "", r.stderr.replace(token, "***") if token else r.stderr)
    del token, url
    # Never leave the credential in the remote — this notebook may get shared.
    subprocess.run(["git", "-C", str(REPO), "remote", "set-url", "origin",
                    "https://github.com/TomOffermann/evolve.git"], check=False)

# --- alternatives -----------------------------------------------------------
# from google.colab import drive; drive.mount("/content/drive")
# REPO = Path("/content/drive/MyDrive/evolve")
#
# from google.colab import files; files.upload()      # evolve.zip
# !unzip -q evolve.zip -d /content && REPO = Path("/content/evolve")
# ----------------------------------------------------------------------------

CODE    = REPO / "code"
RESULTS = CODE / "cpu_benchmark" / "results"
RESULTS.mkdir(parents=True, exist_ok=True)
assert (CODE / "experiments" / "e12_additivity.py").exists(), "e12_additivity.py not found"
print("repo ok:", REPO)
print("branch:", subprocess.run(["git","-C",str(REPO),"rev-parse","--abbrev-ref","HEAD"],
                                capture_output=True, text=True).stdout.strip())

## Step 1 — calibrate the instrument

Before trusting any number, check the readout can reproduce itself. This evaluates the **same**
population twice and correlates the results.

Expect the binary readout (what training uses) to score **0.00 at σ = 0.005** — which problems a
member solves is pure sampling noise at that step size. The `exact` readout computes
$q_{ib} = \sum_{s \in S_b} \prod_t p_i(s_t \mid \cdot)$ in closed form instead of sampling it, so
it is noiseless by construction. Everything downstream uses `exact`.

In [ ]:
%cd -q {CODE}
!python experiments/e12_additivity.py --calibrate \
    --n-pop 256 --sigmas 0.005 0.05 0.2 --crn-sweep 1 4 16 \
    --device {DEVICE} --output {RESULTS} 2>&1 | grep -v "UserWarning\|searchsorted"

## Step 2 — does `u_p` replicate?

`part_cos` = mean over parts of $\cos(u_p^{(A)}, u_p^{(B)})$ for two independently drawn
populations. This is the load-bearing measurement.

Read the **σ column against the N column**. If this were a sample-size problem, quadrupling `N`
would fix it. It does not — that is the $O(\sigma^2)$ signal against $O(\sigma)$ noise, and it is
the result that forces the two-timescale design.

In [ ]:
N_POP   = "256 1024" if DEVICE == "cuda" else "256 1024"   # drop to "256" on slow CPU
REPEATS = 3 if DEVICE == "cuda" else 2

%cd -q {CODE}
!python experiments/e12_probe_part_replication.py \
    --parts 4 8 --n-pop {N_POP} --sigmas 0.005 0.02 0.05 0.2 \
    --repeats {REPEATS} --device {DEVICE} --output {RESULTS} \
    2>&1 | grep -v "UserWarning\|searchsorted"

### Plot — where the part effect becomes estimable

In [ ]:
import json
import matplotlib.pyplot as plt
from matplotlib.ticker import FixedLocator, FixedFormatter

SURFACE, INK, INK2, MUTED = "#fcfcfb", "#0b0b0b", "#52514e", "#a3a29c"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a"]          # validated: CVD dE 9.2, normal dE 24.0
TRAIN_SIGMA, ESTIMABLE = 0.005, 0.60

rows = json.load(open(RESULTS / "e12_probe_seed0.json"))["rows"]
Ps = sorted({r["P"] for r in rows})
Ns = sorted({r["n_pop"] for r in rows})

fig, axes = plt.subplots(1, len(Ps), figsize=(5.6 * len(Ps), 4.4),
                         sharey=True, facecolor=SURFACE)
axes = [axes] if len(Ps) == 1 else list(axes)

for ax, P in zip(axes, Ps):
    ax.set_facecolor(SURFACE)
    ax.axhspan(ESTIMABLE, 1.0, color=SERIES[2], alpha=0.07, lw=0, zorder=0)
    ax.axhline(0, color=MUTED, lw=1, zorder=1)
    ax.axvline(TRAIN_SIGMA, color=MUTED, lw=1, ls=(0, (4, 3)), zorder=1)

    for i, N in enumerate(Ns):
        sub = sorted([r for r in rows if r["P"] == P and r["n_pop"] == N],
                     key=lambda r: r["sigma"])
        if not sub:
            continue
        xs = [r["sigma"] for r in sub]
        ys = [r["part_cos"] for r in sub]
        es = [r.get("sem", 0.0) for r in sub]
        c = SERIES[i % len(SERIES)]
        ax.errorbar(xs, ys, yerr=es, color=c, lw=2, marker="o", ms=8,
                    mec=SURFACE, mew=2, capsize=0, elinewidth=1.5,
                    zorder=3, label=f"N = {N}")
        # Direct label: the aqua slot sits under 3:1 on this surface, so identity
        # never rests on colour alone.
        ax.annotate(f"N={N}  ({N // P}/part)", (xs[-1], ys[-1]),
                    textcoords="offset points", xytext=(9, 0), va="center",
                    fontsize=9.5, color=INK2, zorder=4)

    ax.set_xscale("log")
    sig = sorted({r["sigma"] for r in rows})
    ax.xaxis.set_major_locator(FixedLocator(sig))
    ax.xaxis.set_major_formatter(FixedFormatter([f"{s:g}" for s in sig]))
    ax.set_xlim(min(sig) * 0.6, max(sig) * 3.4)
    ax.set_ylim(-0.6, 1.05)
    ax.set_title(f"P = {P} parts", fontsize=12, color=INK, pad=10, loc="left")
    ax.set_xlabel("perturbation scale  σ", fontsize=10.5, color=INK2)
    ax.grid(axis="y", color=MUTED, alpha=0.25, lw=0.8)
    ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(MUTED)
    ax.tick_params(colors=INK2, labelsize=9.5)

axes[0].set_ylabel("part_cos   (replication of $u_p$)", fontsize=10.5, color=INK2)
axes[0].annotate("estimable", (0.02, ESTIMABLE + 0.04), xycoords=("axes fraction", "data"),
                 fontsize=9.5, color=INK2)
axes[0].annotate("training σ", (TRAIN_SIGMA, -0.55), rotation=90, fontsize=9,
                 color=INK2, ha="right", va="bottom")
axes[0].legend(frameon=False, fontsize=9.5, labelcolor=INK2, loc="upper left")
fig.suptitle("The part effect is second-order: σ moves it, N does not",
             fontsize=13.5, color=INK, x=0.005, ha="left", y=1.02)
fig.tight_layout()
plt.show()

print(f"\n{'P':>4} {'N':>6} {'per-part':>9} {'sigma':>8} {'part_cos':>10} {'sem':>7}")
for r in sorted(rows, key=lambda r: (r["P"], r["n_pop"], r["sigma"])):
    print(f"{r['P']:>4} {r['n_pop']:>6} {r['per_part']:>9} {r['sigma']:>8.4g} "
          f"{r['part_cos']:>+10.3f} {r.get('sem', 0):>7.3f}")

## Step 3 — additivity, at a σ where the question is answerable

Only worth running where Step 2 showed `part_cos` well above 0. Running this at σ = 0.005 divides
by a zero ceiling and returns garbage — the script refuses and prints `NO SIGNAL`.

`additivity` = `part_cos(k) / part_cos(1)`. The denominator is measured on an *independent*
population, so it is the replication ceiling of the estimator itself, not 1.0 by construction.

- **additivity ≈ 1 across k** → superposition holds; one rollout really does carry k hypotheses.
- **additivity falling in k** → cross terms dominate; decoding is dead and everything downstream
  collapses to per-part means at k = 1.
- **shuffled ≉ 0** → the pipeline is broken; ignore every other number.

In [ ]:
%cd -q {CODE}
!python experiments/e12_additivity.py --readout exact \
    --parts 8 --n-active 1 2 4 --sigmas 0.05 0.2 \
    --n-pop 1024 --repeats {REPEATS} --device {DEVICE} --output {RESULTS} \
    2>&1 | grep -v "UserWarning\|searchsorted"

### Plot — how fast superposition decays

In [ ]:
import json
from collections import defaultdict
import matplotlib.pyplot as plt
from matplotlib.ticker import FixedLocator, FixedFormatter

rows = json.load(open(RESULTS / "e12_additivity_exact_seed0_g0.json"))["rows"]
sigmas = sorted({r["sigma"] for r in rows})
ks = sorted({r["n_active"] for r in rows})

fig, ax = plt.subplots(figsize=(7.2, 4.6), facecolor=SURFACE)
ax.set_facecolor(SURFACE)
ax.axhline(1.0, color=MUTED, lw=1, ls=(0, (4, 3)), zorder=1)
ax.annotate("perfect superposition", (ks[-1], 1.0), textcoords="offset points",
            xytext=(0, 7), ha="right", fontsize=9.5, color=INK2)
ax.axhline(0, color=MUTED, lw=1, zorder=1)

for i, sg in enumerate(sigmas):
    by_k = defaultdict(list)
    for r in rows:
        if r["sigma"] == sg:
            by_k[r["n_active"]].append(r["part_cos"])
    if 1 not in by_k:
        continue
    ceil = sum(by_k[1]) / len(by_k[1])
    xs = [k for k in ks if k in by_k]
    ys = [(sum(by_k[k]) / len(by_k[k])) / ceil for k in xs]
    c = SERIES[i % len(SERIES)]
    ax.plot(xs, ys, color=c, lw=2, marker="o", ms=8, mec=SURFACE, mew=2,
            zorder=3, label=f"σ = {sg:g}")
    ax.annotate(f"σ={sg:g}", (xs[-1], ys[-1]), textcoords="offset points",
                xytext=(9, 0), va="center", fontsize=9.5, color=INK2, zorder=4)

    shuf = defaultdict(list)
    for r in rows:
        if r["sigma"] == sg:
            shuf[r["n_active"]].append(r["part_cos_shuffled"] / ceil)
    sx = [k for k in ks if k in shuf]
    ax.plot(sx, [sum(shuf[k]) / len(shuf[k]) for k in sx], color=c, lw=1.4,
            ls=(0, (2, 2)), marker=None, alpha=0.55, zorder=2)

ax.set_xscale("log", base=2)
ax.xaxis.set_major_locator(FixedLocator(ks))
ax.xaxis.set_major_formatter(FixedFormatter([str(k) for k in ks]))
ax.set_xlim(min(ks) * 0.85, max(ks) * 1.6)
ax.set_ylim(-0.2, 1.3)
ax.set_xlabel("n_active   (parts perturbed per member)", fontsize=10.5, color=INK2)
ax.set_ylabel("additivity   (part_cos(k) / ceiling)", fontsize=10.5, color=INK2)
ax.set_title("Superposition decay — solid: measured, dashed: shuffled-label control",
             fontsize=12, color=INK, loc="left", pad=10)
ax.grid(axis="y", color=MUTED, alpha=0.25, lw=0.8)
ax.set_axisbelow(True)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
for s in ("left", "bottom"):
    ax.spines[s].set_color(MUTED)
ax.tick_params(colors=INK2, labelsize=9.5)
ax.legend(frameon=False, fontsize=9.5, labelcolor=INK2, loc="lower left")
fig.tight_layout()
plt.show()

print(f"\n{'sigma':>8} {'n_act':>6} {'part_cos':>10} {'shuffled':>10} "
      f"{'in_regime':>10} {'transfer':>9} {'rho_parts':>10}")
for r in sorted(rows, key=lambda r: (r["sigma"], r["n_active"])):
    print(f"{r['sigma']:>8.4g} {r['n_active']:>6} {r['part_cos']:>+10.3f} "
          f"{r['part_cos_shuffled']:>+10.3f} {r['in_regime']:>+10.3f} "
          f"{r['transfer']:>+9.3f} {r['rho_parts_vs_heldout']:>+10.3f}")

## What to change next

**The control that matters.** Nothing above shows the partition estimated at probe σ is the *right*
partition at training σ. That is the next experiment and it is not optional: learn a partition from
probe-σ signatures, then compare it against a **random** partition at matched `P`, matched
`n_active` and matched merge/split schedule, with σ swept. If learned ≈ random, the E8/E9 effect is
drift control and the honest write-up is the drift paper.

**Does additivity survive training?** Everything here is at gen 0, post-SFT. The curvature-to-
gradient ratio $\mathrm{tr}(H|_p)/\lVert g|_p\rVert$ can move. Add `--train-gens 150` to the Step 3
cell — it advances the policy with the E8 partitioned arm first, then measures.

**Seeds.** One seed throughout. `--seed 1 2 3` and pool before believing any of it.

Reference columns, if you need them mid-run:

| column | means |
|---|---|
| `part_cos` | replication of $u_p$ across independent populations — the load-bearing number |
| `part_cos_shuffled` | same with part labels permuted; must be ≈ 0 |
| `in_regime` | how much the **mask** explains out-of-sample. Low is expected — the continuous half of the genotype (which direction inside the part) does the work the discrete half cannot. **Not** a measure of additivity |
| `transfer` | does the isolated model predict combined behaviour |
| `rho_parts_vs_heldout` | E0's test one level up: does signature distance predict held-out behavioural decorrelation, on an independent population |